# MBPP Dataset Transformation

This notebook transforms the MBPP (Mostly Basic Python Problems) dataset into a structured format suitable for training and evaluation.

## Output Format
- **task_id**: Original task identifier
- **text**: Problem description
- **example_test_input**: Parameters from the first test case (serves as an example)
- **example_test_output**: Expected output from the first test case
- **test_input**: Parameters from subsequent test cases
- **test_output**: Expected outputs formatted as `print()` would display them

## Prerequisites
- `datasets` library for loading MBPP
- `pandas` for data manipulation
- `ast` for safe expression evaluation

In [ ]:
# Configuration
DATA_CACHE_DIR = "..//data"
OUTPUT_DIR = "./dataset"
OUTPUT_FILE_EXAMPLES = f"{OUTPUT_DIR}/mbpp_jitgen_validation.csv"

In [48]:
# Imports
import re
import ast
import sys
import io
import pandas as pd
from datasets import load_dataset

In [49]:
# Load MBPP dataset
dataset_full = load_dataset(
    "mbpp", cache_dir=DATA_CACHE_DIR, download_mode="reuse_dataset_if_exists"
)
df_original = dataset_full["validation"].to_pandas()

## Utility Functions
Core functions for parsing assertions and handling different data types.

In [50]:
def parse_assert_statement(assert_stmt):
    stmt = assert_stmt.strip().replace("assert ", "")
    if " == " not in stmt:
        return None, None
    parts = stmt.split(" == ", 1)
    if len(parts) != 2:
        return None, None
    function_call = parts[0].strip()
    expected_result = parts[1].strip()
    return function_call, expected_result


def extract_function_parameters(function_call):
    pattern = r"(\w+)\((.*)\)"
    match = re.match(pattern, function_call)
    if not match:
        return function_call
    return match.group(2)


def get_print_output(value):
    old_stdout = sys.stdout
    sys.stdout = captured_output = io.StringIO()
    print(value)
    sys.stdout = old_stdout
    return captured_output.getvalue().strip()


def safe_eval(expression):
    safe_globals = {
        "sys": sys,
        "len": len,
        "sum": sum,
        "max": max,
        "min": min,
        "abs": abs,
        "round": round,
        "__builtins__": {},
    }
    try:
        result = ast.literal_eval(expression)
        return get_print_output(result)
    except:
        try:
            result = eval(expression, safe_globals)
            return get_print_output(result)
        except:
            return expression

In [51]:
def create_dataset_with_examples(original_dataset):
    new_rows = []
    for _, row in original_dataset.iterrows():
        task_id = row["task_id"]
        text = row["text"]
        test_list = row["test_list"]
        parsed_assertions = []
        for assert_stmt in test_list:
            function_call, expected_result = parse_assert_statement(assert_stmt)
            if function_call and expected_result:
                test_input = extract_function_parameters(function_call)
                test_output = safe_eval(expected_result)
                parsed_assertions.append(
                    {"test_input": test_input, "test_output": test_output}
                )
        if len(parsed_assertions) >= 2:
            example_test_input = parsed_assertions[0]["test_input"]
            example_test_output = parsed_assertions[0]["test_output"]
            for assertion in parsed_assertions[1:]:
                new_rows.append(
                    {
                        "task_id": task_id,
                        "text": text,
                        "example_test_input": example_test_input,
                        "example_test_output": example_test_output,
                        "test_input": assertion["test_input"],
                        "test_output": assertion["test_output"],
                    }
                )
    return pd.DataFrame(new_rows)

## Create and Save Dataset with Examples

In [52]:
examples_dataset = create_dataset_with_examples(df_original)
examples_dataset.to_csv(OUTPUT_FILE_EXAMPLES, index=False)

In [53]:
# Save datasets
examples_dataset.to_csv(OUTPUT_FILE_EXAMPLES, index=False)

print("Datasets saved:")
print(f"  With examples: {OUTPUT_FILE_EXAMPLES}")

Datasets saved:
  With examples: ./dataset/mbpp_jitgen_validation.csv


## Results

The transformation successfully creates the examples dataset:

### Examples Dataset (`mbpp_transformed_dataset_with_examples.csv`)
- First assertion serves as example for remaining assertions
- Columns: `task_id`, `text`, `example_test_input`, `example_test_output`, `test_input`, `test_output`

### Key Features
- ✅ Function calls in assertions are properly evaluated
- ✅ Output formatting matches `print()` behavior exactly
- ✅ Complex data types (dicts, tuples, lists) handled correctly
- ✅ Safe evaluation prevents code injection